### helper.py

In [122]:
import re
import yaml
import json
import os

class LiteralString(str):
    pass
def literal_representer(dumper,value):
    return dumper.represent_scalar('tag:yaml.org,2002:str', value, style='|')

# Register custom representer
yaml.add_representer(LiteralString, literal_representer)

class Helper:
    @staticmethod
    def load_file(filepath: str) -> str:
        with open(filepath, "r", encoding="utf-8") as file:
            return file.read()
    @staticmethod
    def load_yaml(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)
    @staticmethod
    def load_json(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    @staticmethod
    def save_yaml(newconfig,filepath: str):
        with open(filepath,"w",encoding="utf-8") as file:
            return yaml.dump(newconfig,file,sort_keys=False)
    @staticmethod
    def fop(num: float) -> float:
        return float(f"{num:.1f}")
    @staticmethod
    def prettyjson(txt:str) -> str:
        return str(json.dumps(txt,indent=4, ensure_ascii=False))
    @staticmethod
    def to_literal(value):
        if isinstance(value,str) and "\n" in value:
            return LiteralString(value)
        return value
    @staticmethod
    def deep_literal_transform(data):
        if isinstance(data, dict):
            return {k: Helper.deep_literal_transform(v) for k,v in data.items()}
        if isinstance(data, list):
            return [Helper.deep_literal_transform(i) for i in data]
        return Helper.to_literal(data)

### Llmcaller.py

In [ ]:
import asyncio
from pydantic import BaseModel, Field, ValidationError
from typing import Dict
from google.genai import types

class CriterionScore(BaseModel):
    score: int = Field(ge=0, le=5)
    feedback: str

class SectionEvaluation(BaseModel):
    section: str
    scores: Dict[str, CriterionScore]
    session_feedback: str

class LlmCaller(Helper):
    def __init__(self):
        self.client    = genai.Client(api_key="aSyBuztp1gdQDlmjT2Ut_Kg7r2V1wFsGycVM")
        self.model_cfg = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")
        self.model     = self.model_cfg["model"]["generation_model"]
        self.usd2bath  = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['currency']['USD_to_THB']
        self.log_digit = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['logging']['logging_round_digit']
    def extract_json(text: str) -> dict:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("NO_JSON_OBJECT_FOUND")
        return json.loads(match.group())
    def _parse(self, resp):
        text = resp.text.strip()
        text = re.sub(r"^```json|```$", "", text).strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError as e:
            pass        
        try:
            return self.extract_json(text)
        except Exception as e:
            raise ValueError(f"INVALID_JSON::{text}") from e
    def _call_raw(self, prompt: str):
        resp = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
    #         config=types.GenerateContentConfig(
    #             # thinking_config=types.ThinkingConfig(thinking_budget=1024)
    #             # Turn off thinking:
    #             thinking_config=types.ThinkingConfig(thinking_budget=0)
    #             # Turn on dynamic thinking:
    #             # thinking_config=types.ThinkingConfig(thinking_budget=-1)
    # )
)
        parsed = self._parse(resp)
        return parsed, resp
    def _validate(self, raw_output:dict)->SectionEvaluation:
        return SectionEvaluation.model_validate(raw_output)
    def _repair_prompt(self, error_msg: str) -> str:
        return f"""
                Your previous response was INVALID.
                Validation error:
                {error_msg}
                STRICT RULES:
                - Return JSON only
                - No markdown
                - No explanation
                - Follow schema exactly
                - Section name must start with a capital letters (e.g. "Education")
                Expected format:
                {{
                    "section": "<Section_name>",
                    "scores": {{
                        "<criterion>": {{ 
                            "score": 0-5, 
                            "feedback": "string" 
                        }}
                    }},
                    "session_feedback":"string"
                }}
        """
    def call(self,prompt:str, max_retry:int = 3):
        last_error = None
        repair_prompt = "\n"
        for attemp in range(max_retry):
            final_prompt = repair_prompt + prompt
            # print(f"final_prompt attemp : {attemp} -> \n {final_prompt}")
            # print(f"Output -> \n{output}")
            try:
                output, raw = self._call_raw(final_prompt)
                validated   = self._validate(output)
                # print('Status : 1')
                return validated.model_dump(),raw
            except (ValidationError, ValueError) as e:
                last_error = str(e)
                repair_prompt = self._repair_prompt(last_error)
                # print('Status : 0')
                print("Output error recall again ...")
            finally:
                print('='*100)
        return {
            "section":"UNKNOW",
            "scores":{}
        },raw
    async def call_async(self, prompt: str):
        return await asyncio.to_thread(self.call, prompt)

### PromptBuilder.py

In [124]:
from google import genai
import json
import os
import yaml


class BasePromptBuilder(Helper):
    '''
    PromptBuilder v3 : PromptBuilder + Session,Global feedback + PromptSplit
    '''
    base_dir = r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts"         # Prompts .yaml config file folder path
    def __init__(self, section, criteria, targetrole, cvresume, include_fewshot: bool = True, output_lang = "en"):
        self.section         = section
        self.criteria        = criteria[::-1]
        self.targetrole      = targetrole
        self.cvresume        = cvresume
        self.include_fewshot = include_fewshot
        self.output_lang     = output_lang

        self.global_config   = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts\base.yaml")
        self.section_config  = self.load_yaml(f"{self.base_dir}/{self.section.lower()}.yaml")
        self.number_of_words = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")["output"]["number_of_words"]
        self.criteria_cfg    = self.section_config['criteria']
    def _build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            },
            "session_feedback":""
        }
    
    def _build_criteria_block(self) -> str:
        blocks = []
        for crit in self.criteria:
            block = f"- {crit}\n"
            for i in [5,3,1]:
                block += f"  score {i} :\n"
                x = f"score{i}"
                block += "\n".join(
                    "    " + line
                    for line in self.criteria_cfg[crit][x].splitlines()
                ) + "\n"
            blocks.append(block)
        return "".join(blocks)

    def build(self):
        config_role       = self.global_config['role']['role1']
        config_task       = self.section_config['task']['task1']
        config_lang       = self.global_config['Language_output_style'][self.output_lang]
        config_expected   = self.section_config['expected_content'][self.section]
        criteria_block    = self._build_criteria_block()
        config_example    = self.section_config['output_guidelines'][self.section]
        config_scale      = self.global_config['scale']['score1']
        config_feedback   = self.global_config['feedback']['globalfeedback']

        prompt_role       = f"Role :\n{config_role}\n\n"
        prompt_task       = f"Task :\n{config_task}\n"
        prompt_lang       = f"Output language instruction :\n{config_lang}\n"
        prompt_expected   = f"Expected :\n{config_expected}\n"
        prompt_criteria   = f"Criteria :\n{criteria_block}\n"
        prompt_scale      = f"Scale :\n{config_scale}\n"
        prompt_feedback   = f"Session feedback :\n{config_feedback}\n\n"
        prompt_Op_example = f"Output guideline :\n{config_example}\n\n"
        prompt_Op_format  = f"Output format :\n{json.dumps(self._build_response_template(), indent=2)}\n\n"
        prompt_cvresume   = f"CV/Resume :\n{self.cvresume}\n"

        prompt = (
            prompt_role + prompt_task + prompt_lang
            + prompt_expected + prompt_criteria + prompt_scale + prompt_feedback 
            + prompt_Op_example + prompt_Op_format + prompt_cvresume 
        )

        prompt = prompt.replace("<section_name>", self.section)
        prompt = prompt.replace("<targetrole>", self.targetrole)
        prompt = prompt.replace("<number_of_words>", str(self.number_of_words))

        return prompt

In [125]:
op1

{'section': 'Profile',
 'scores': {'ContentQuality': {'score': 1,
   'feedback': "This section lacks a professional title, failing to clearly establish the candidate's professional identity."},
  'Completeness': {'score': 3,
   'feedback': 'It provides essential contact information and links, but a professional title is missing.'}},
 'session_feedback': "The profile clearly lists contact details and professional links. However, the absence of an explicit professional title in this section limits immediate understanding of the candidate's core identity for the data scientist role."}

In [126]:
s1

{'section': 'Profile',
 'total_section_raw_score': 8.0,
 'total_section_max_score': 20.0,
 'scores': {'ContentQuality': {'score': 2.0,
   'feedback': "This section lacks a professional title, failing to clearly establish the candidate's professional identity."},
  'Completeness': {'score': 6.0,
   'feedback': 'It provides essential contact information and links, but a professional title is missing.'}},
 'session_feedback': "The profile clearly lists contact details and professional links. However, the absence of an explicit professional title in this section limits immediate understanding of the candidate's core identity for the data scientist role."}

In [127]:
s2

{'section': 'Summary',
 'total_section_raw_score': 48.0,
 'total_section_max_score': 50.0,
 'scores': {'RoleRelevance': {'score': 10.0,
   'feedback': 'The summary directly aligns with the data scientist role, highlighting core skills and relevant application areas.'},
  'Length': {'score': 10.0,
   'feedback': 'The summary is concise and falls perfectly within the recommended 2-4 sentence length.'},
  'Grammar': {'score': 10.0,
   'feedback': 'The writing is clear, professional, and free of grammatical errors, enhancing readability.'},
  'ContentQuality': {'score': 8.0,
   'feedback': "It uses specific language about focus areas and business impact, though 'passionate' is a minor cliché."},
  'Completeness': {'score': 10.0,
   'feedback': "The summary clearly establishes the candidate's identity, expertise, and value proposition for the role."}},
 'session_feedback': "This summary effectively positions the candidate for a Data Scientist role by clearly stating expertise in data analys

In [128]:
from datetime import datetime,timezone,timedelta
# from core.helper import Helper
# from core.llmcaller import LlmCaller
import copy
import json

class SectionScoreAggregator(Helper):
    def __init__(self):
        self.weight_config          = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")         # config
        
    def aggregate(self,llm_output:dict):
        '''
        Convert, reshape, transform output format that we got from llm
        '''
        self.llm_output   = llm_output               # Op
        self.section_name = llm_output["section"]    # Get section name such as Profile, Summary, ..., Skills
        self.section_config_scores   = self.weight_config["weights"][self.section_name]   # Get max score in weight.yaml config
        scaled_criteria_scores  = {}            # Output dictionary
        total_section_raw_score = 0.0           # Accumulate raw score from every criteria
        total_section_max_score = 0.0           # Accumulate maximum score from every criteria that posible
        scores_copy  = copy.deepcopy(llm_output["scores"])         # Protect multiple mutation when we run more than one time
        for criteria,body in scores_copy.items():  # Loop with op...
            raw_llm_score = body["score"]       # raw score for every criteria each section
            if raw_llm_score == 0:              # If LLM detect empty value then max_score should be 0 (don't calcualte it)
                max_score_from_config = 0       # score = 0 instead maximum score
            else:
                max_score_from_config = self.section_config_scores[criteria] # Max score in weight.yaml (default=10)
            scaled_score = (raw_llm_score / 5) * max_score_from_config  # raw_score/max scale score(5) x max score in weight.yaml(10)
            body["score"] = scaled_score            # Replace new scaled score in body
            scaled_criteria_scores[criteria] = body # Create new dict (Op -> S)
            total_section_raw_score = total_section_raw_score + scaled_score # Accumulate scaled raw score from each criteria in each seciton
            total_section_max_score = total_section_max_score + max_score_from_config # Accumulate full score from config file that Llm not detect 0
        return {
                "section": self.section_name,
                "total_section_raw_score":total_section_raw_score,
                "total_section_max_score":total_section_max_score,
                "scores":scaled_criteria_scores,
                "session_feedback":self.llm_output['session_feedback']
            }
    
class GlobalAggregator(LlmCaller,Helper):
    def __init__(self,SectionScoreAggregator_output:list,output_lang):
        super().__init__()    # Run Llmcaller class 
        self.section_outputs = SectionScoreAggregator_output
        self.timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
        self.model_config    = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
        self.weight_config   = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
        self.prompt_config   = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml")    # includes prompt version
        self.config_lang     = self.prompt_config['Language_output_style'][output_lang]
        
    def normalize_score_to_grade(self, total_raw_score, total_max_score):
        # print(f"total_raw_score -> {total_raw_score}")
        # print(f"total_max_score -> {total_max_score}")
        if total_max_score == 0:
            return "Grading error"
        normalize_score = (total_raw_score / total_max_score) * 100
        # print(f"normalize_score -> {normalize_score}/100")
        score = round(normalize_score)
        if score == 0:
            return "-"
        elif score == 100:
            return "S"
        elif 95 <= score <= 99:
            return "A+"
        elif 90 <= score <= 94:
            return "A"
        elif 85 <= score <= 89:
            return "A-"
        elif 80 <= score <= 84:
            return "B+"
        elif 75 <= score <= 79:
            return "B"
        elif 70 <= score <= 74:
            return "B-"
        elif 65 <= score <= 69:
            return "C+"
        elif 60 <= score <= 64:
            return "C"
        elif 55 <= score <= 59:
            return "C-"
        elif 50 <= score <= 54:
            return "D+"
        elif 45 <= score <= 49:
            return "D"
        elif 40 <= score <= 44:
            return "D-"
        elif 1 <= score <= 39:
            return "F"
        else:
            return "Grading error"
        
    def aggregate_weighted_section_scores(self):         # def fn1(self):
        weights      = self.weight_config["weights"]
        contribution = {}                    # Keep stat of score and detail
        total_weighted_raw_score = 0.0       # Accumulate raw score after time by weight
        total_weighted_max_score = 0.0       # Accumulate max score after time by weight
        for section_data in self.section_outputs: # Loop with section_output (Ss)
            section_name            = section_data["section"]   # Get section name
            total_section_raw_score = section_data["total_section_raw_score"]    # Get raw section score from each section
            total_section_max_score = section_data["total_section_max_score"]    # Get max section score from each section
            section_weight          = weights[section_name]["section_weight"]    # Get section_weight from weight.yaml e.g. 0.1,0.2
            
            total_section_raw_score_x_weight = total_section_raw_score*section_weight   # raw_score x weight
            total_section_max_score_x_weight = total_section_max_score*section_weight   # max_score x weight

            contribution[section_name] = {
                "session_grade":self.normalize_score_to_grade(total_section_raw_score_x_weight,total_section_max_score_x_weight), # Grading CVResume with (total_section_raw_score_x_weight/total_section_max_score_x_weight)*100 -> if else 
                "total_section_raw_score":total_section_raw_score,
                "total_section_max_score":total_section_max_score,
                "section_weight":section_weight,
                "total_section_raw_score_x_weight":total_section_raw_score_x_weight,
                "total_section_max_score_x_weight":total_section_max_score_x_weight
            }
            total_weighted_raw_score  = total_weighted_raw_score + total_section_raw_score_x_weight   # E(raw_score x weight)
            total_weighted_max_score  = total_weighted_max_score + total_section_max_score_x_weight   # E(max_score x weight)
            
        return {
            "global_grade":self.normalize_score_to_grade(total_weighted_raw_score,total_weighted_max_score), # Grading CVResume with (total_weighted_raw_score/total_weighted_max_score)*100 -> if else 
            "total_weighted_raw_score":total_weighted_raw_score, # E(raw_score x weight) Summation of raw score for every section every criteria
            "total_weighted_max_score":total_weighted_max_score, # E(max_score x weight) Summation of max score for every section every criteria
            "section_contribution":contribution,                 # Details
            "globalfeedback":self.parse_global_feedback
        }
    
    def fn2(self):
        details = {}
        for section_data in self.section_outputs:
            print(f"section_data->\n{section_data}")
            details[section_data["section"]] = {
                'total_section_raw_score':section_data['total_section_raw_score'],
                'total_section_max_score':section_data['total_section_max_score'],
                'scores':section_data['scores'],
                'section_feedbak':section_data['session_feedback']
            }
        # prompt = self.prompt_config['feedback']['globalfeedback']
        prompt = f'''
            You are an expert CV and Resume reviewer.

            Your task is to generate a SINGLE, GLOBAL feedback summary based on the full resume evaluation results below.

            IMPORTANT:
            - You MUST read and consider feedback from ALL resume sections.
            - Do NOT repeat section-by-section feedback.
            - Synthesize insights into an overall assessment.
            - Assume the user will NOT read individual section details.
            - Limit the session_feedback to one short paragraph with 20 words.

            Focus on:
            1. Overall strengths of the resume
            2. Key weaknesses or gaps
            3. High-impact, actionable improvement advice

            Guidelines:
            - Be professional, constructive, and specific
            - Avoid generic statements
            - Do NOT assume missing information
            - Base your feedback ONLY on the evaluation data provided
            {self.config_lang}

            INPUT (section-level evaluation results):
            {json.dumps(details, indent=2)}

            STRICT OUTPUT RULES:
            - Return JSON ONLY
            - No markdown
            - No explanation
            - No extra text

            Output schema:
            {
                {
                "response": "Concise but insightful global feedback covering strengths, weaknesses, and improvement suggestions."
                }
            }
        '''
        # print(f"prompt->\n{prompt}")
        self.parse_global_feedback,_ = self._call_raw(prompt)
        # print(self.parse)
        return details
    
    def fn3(self):
        return {
            "model_name": self.model_config['model']['generation_model'],
            "timestamp": self.timestamp,
            "weights_version": self.weight_config.get("version", "unknown"),
            "prompt_version": self.prompt_config.get("version", "unknown")
        }
    
    def fn0(self):
        detail_part     = self.fn2()
        conclution_part = self.aggregate_weighted_section_scores()
        metadata_part   = self.fn3()
        
        return {
            "Conclution":conclution_part,
            "Section_detail":detail_part,
            "Metadata":metadata_part
        }
        

In [129]:
s2

{'section': 'Summary',
 'total_section_raw_score': 48.0,
 'total_section_max_score': 50.0,
 'scores': {'RoleRelevance': {'score': 10.0,
   'feedback': 'The summary directly aligns with the data scientist role, highlighting core skills and relevant application areas.'},
  'Length': {'score': 10.0,
   'feedback': 'The summary is concise and falls perfectly within the recommended 2-4 sentence length.'},
  'Grammar': {'score': 10.0,
   'feedback': 'The writing is clear, professional, and free of grammatical errors, enhancing readability.'},
  'ContentQuality': {'score': 8.0,
   'feedback': "It uses specific language about focus areas and business impact, though 'passionate' is a minor cliché."},
  'Completeness': {'score': 10.0,
   'feedback': "The summary clearly establishes the candidate's identity, expertise, and value proposition for the role."}},
 'session_feedback': "This summary effectively positions the candidate for a Data Scientist role by clearly stating expertise in data analys

<hr>

In [130]:
caller    = LlmCaller()
agg       = SectionScoreAggregator()
mock_data = Helper.load_json(r"C:\Users\TunKedsaro\Desktop\CVResume\src\mock\resume4.json")

In [131]:
p1 = BasePromptBuilder(
    section     = "Profile",
    criteria    = ["Completeness", "ContentQuality"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt1 = p1.build()
p3 = BasePromptBuilder( 
    section     = "Education", 
    criteria    = ["Completeness","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt3 = p3.build()

op1,raw1 = caller.call(prompt1)
op3,raw3 = caller.call(prompt3)

In [132]:
op1

{'section': 'Profile',
 'scores': {'ContentQuality': {'score': 0,
   'feedback': 'The profile section does not explicitly state a professional title or current status.'},
  'Completeness': {'score': 3,
   'feedback': 'Basic identification and contact information are present, but a professional title is missing.'}},
 'session_feedback': 'The profile provides contact details and links, but omitting a professional title hinders immediate identification for the Data Scientist role.'}

In [133]:
op3

{'section': 'Education',
 'scores': {'RoleRelevance': {'score': 5,
   'feedback': 'The educational background strongly supports the knowledge and skills expected for the target role.'},
  'Completeness': {'score': 5,
   'feedback': "Clearly presents institution, degree, field of study, and dates, allowing the reader to easily understand the candidate's education."}},
 'session_feedback': "This education section is exceptionally strong, featuring two highly relevant master's degrees directly in Data Science and Statistical Information Processing, with comprehensive coursework. All academic details are fully provided. This robust academic background significantly enhances the candidate's positioning for a data scientist role, demonstrating deep foundational expertise."}

In [134]:
s1 = agg.aggregate(op1)
s2 = agg.aggregate(op3)

In [135]:
s1

{'section': 'Profile',
 'total_section_raw_score': 6.0,
 'total_section_max_score': 10.0,
 'scores': {'ContentQuality': {'score': 0.0,
   'feedback': 'The profile section does not explicitly state a professional title or current status.'},
  'Completeness': {'score': 6.0,
   'feedback': 'Basic identification and contact information are present, but a professional title is missing.'}},
 'session_feedback': 'The profile provides contact details and links, but omitting a professional title hinders immediate identification for the Data Scientist role.'}

In [136]:
s2

{'section': 'Education',
 'total_section_raw_score': 20.0,
 'total_section_max_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0,
   'feedback': 'The educational background strongly supports the knowledge and skills expected for the target role.'},
  'Completeness': {'score': 10.0,
   'feedback': "Clearly presents institution, degree, field of study, and dates, allowing the reader to easily understand the candidate's education."}},
 'session_feedback': "This education section is exceptionally strong, featuring two highly relevant master's degrees directly in Data Science and Statistical Information Processing, with comprehensive coursework. All academic details are fully provided. This robust academic background significantly enhances the candidate's positioning for a data scientist role, demonstrating deep foundational expertise."}

In [137]:
x = GlobalAggregator(SectionScoreAggregator_output = [s1,s2],output_lang="en")

output = x.fn0()

section_data->
{'section': 'Profile', 'total_section_raw_score': 6.0, 'total_section_max_score': 10.0, 'scores': {'ContentQuality': {'score': 0.0, 'feedback': 'The profile section does not explicitly state a professional title or current status.'}, 'Completeness': {'score': 6.0, 'feedback': 'Basic identification and contact information are present, but a professional title is missing.'}}, 'session_feedback': 'The profile provides contact details and links, but omitting a professional title hinders immediate identification for the Data Scientist role.'}
section_data->
{'section': 'Education', 'total_section_raw_score': 20.0, 'total_section_max_score': 20.0, 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'The educational background strongly supports the knowledge and skills expected for the target role.'}, 'Completeness': {'score': 10.0, 'feedback': "Clearly presents institution, degree, field of study, and dates, allowing the reader to easily understand the candidate's educatio

In [138]:
print(Helper.prettyjson(output))

{
    "Conclution": {
        "global_grade": "A",
        "total_weighted_raw_score": 4.6,
        "total_weighted_max_score": 5.0,
        "section_contribution": {
            "Profile": {
                "session_grade": "C",
                "total_section_raw_score": 6.0,
                "total_section_max_score": 10.0,
                "section_weight": 0.1,
                "total_section_raw_score_x_weight": 0.6000000000000001,
                "total_section_max_score_x_weight": 1.0
            },
            "Education": {
                "session_grade": "S",
                "total_section_raw_score": 20.0,
                "total_section_max_score": 20.0,
                "section_weight": 0.2,
                "total_section_raw_score_x_weight": 4.0,
                "total_section_max_score_x_weight": 4.0
            }
        },
        "globalfeedback": {
            "response": "Strong academic background in Data Science is excellent; add a professional title to your profile 

In [139]:
p1 = BasePromptBuilder(
    section     = "Profile",
    criteria    = ["Completeness", "ContentQuality"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt1 = p1.build()

p2 = BasePromptBuilder( 
    section     = "Summary", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt2 = p2.build()

p3 = BasePromptBuilder( 
    section     = "Education", 
    criteria    = ["Completeness","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt3 = p3.build()

p4 = BasePromptBuilder( 
    section     = "Experience", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt4 = p4.build()

p5 = BasePromptBuilder( 
    section     = "Activities", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt5 = p5.build()

p6 = BasePromptBuilder( 
    section     = "Skills", 
    criteria    = ["Completeness","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt6 = p6.build()

op1,raw1 = caller.call(prompt1)
op2,raw2 = caller.call(prompt2)
op3,raw3 = caller.call(prompt3)
op4,raw4 = caller.call(prompt4)
op5,raw5 = caller.call(prompt5)
op6,raw6 = caller.call(prompt6)

s1 = agg.aggregate(op1)
s2 = agg.aggregate(op2)
s3 = agg.aggregate(op3)
s4 = agg.aggregate(op4)
s5 = agg.aggregate(op5)
s6 = agg.aggregate(op6)

x = GlobalAggregator(
    SectionScoreAggregator_output = [s1,s2,s3,s4,s5,s6],
    output_lang = "en"
    )

output = x.fn0()


section_data->
{'section': 'Profile', 'total_section_raw_score': 8.0, 'total_section_max_score': 20.0, 'scores': {'ContentQuality': {'score': 2.0, 'feedback': "The profile lacks a clear professional title or status, hindering immediate understanding of the candidate's identity."}, 'Completeness': {'score': 6.0, 'feedback': "The section includes contact details and links, but it omits the candidate's professional title or status."}}, 'session_feedback': 'Contact details and links are good, but lacking a professional title prevents immediate role clarity for a data scientist position.'}
section_data->
{'section': 'Summary', 'total_section_raw_score': 50.0, 'total_section_max_score': 50.0, 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'The summary directly highlights expertise in data analysis, experimentation, user behavior, and business impact.'}, 'Length': {'score': 10.0, 'feedback': 'The summary is concise and impactful, adhering perfectly to the recommended length.'}, 'Gram

In [144]:
print(Helper.prettyjson(output))

{
    "Conclution": {
        "global_grade": "A-",
        "total_weighted_raw_score": 29.800000000000004,
        "total_weighted_max_score": 35.0,
        "section_contribution": {
            "Profile": {
                "session_grade": "D-",
                "total_section_raw_score": 8.0,
                "total_section_max_score": 20.0,
                "section_weight": 0.1,
                "total_section_raw_score_x_weight": 0.8,
                "total_section_max_score_x_weight": 2.0
            },
            "Summary": {
                "session_grade": "S",
                "total_section_raw_score": 50.0,
                "total_section_max_score": 50.0,
                "section_weight": 0.1,
                "total_section_raw_score_x_weight": 5.0,
                "total_section_max_score_x_weight": 5.0
            },
            "Education": {
                "session_grade": "S",
                "total_section_raw_score": 20.0,
                "total_section_max_score": 20.